In [ ]:
# =============================================================================
# YouTube NLP Analizi – Türkiye Dijitalleşme Algısı
# İyileştirilmiş sürüm: ~8.000 yorum toplama + temizleme + duygu + TF-IDF + LDA
# =============================================================================
# Kurulum:
# pip install google-api-python-client pandas numpy matplotlib seaborn wordcloud scikit-learn gensim pyLDAvis nltk snowballstemmer transformers torch tqdm
#
# Çalıştırma:
# 1) API anahtarını ortam değişkeni olarak ver:
#    Windows PowerShell:
#    $env:YOUTUBE_API_KEY="BURAYA_API_KEY"
#    python youtube_nlp_iyilestirilmis.py
#
# 2) Ya da aşağıdaki YOUTUBE_API_KEY değişkenine doğrudan yaz.
# =============================================================================

import os
import re
import time
import warnings
from collections import Counter
from datetime import datetime

warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm

from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

from wordcloud import WordCloud

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import silhouette_score

from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

try:
    import nltk
    nltk.download("stopwords", quiet=True)
    from nltk.corpus import stopwords
    NLTK_STOPWORDS = set(stopwords.words("turkish"))
except Exception:
    NLTK_STOPWORDS = set()

try:
    from snowballstemmer import stemmer as SnowballStemmer
    TR_STEMMER = SnowballStemmer("turkish")
    USE_STEMMER = True
except Exception:
    TR_STEMMER = None
    USE_STEMMER = False

try:
    from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
    TRANSFORMERS_OK = True
except Exception:
    TRANSFORMERS_OK = False


# =============================================================================
# 1. AYARLAR
# =============================================================================

YOUTUBE_API_KEY = ""
# İstersen buraya doğrudan yaz:
# YOUTUBE_API_KEY = "API_KEY"

OUTPUT_DIR = "gorseller"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RAW_CSV = "youtube_ham.csv"
CLEAN_CSV = "youtube_temiz.csv"
FINAL_CSV = os.path.join(OUTPUT_DIR, "youtube_nlp_tam.csv")

TARGET_COMMENTS = 8000

# 8 bine yaklaşmak için artırıldı.
# Search API pahalıdır: search.list = 100 quota unit, commentThreads.list = 1 quota unit.
MAX_VIDEOS_PER_KEYWORD = 50
MAX_COMMENTS_PER_VIDEO = 250

# Tek seferde API patlamasın diye bekleme.
REQUEST_SLEEP = 0.15

KEYWORDS = [
    "dijitalleşme Türkiye",
    "Türkiye dijital dönüşüm",
    "dijital uçurum Türkiye",
    "e-devlet Türkiye",
    "dijital eğitim Türkiye",
    "uzaktan eğitim Türkiye",
    "yapay zeka Türkiye",
    "teknoloji Türkiye",
    "internet kullanımı Türkiye",
    "dijital beceri Türkiye",
    "TÜİK dijitalleşme",
    "Türkiye teknoloji gelişimi",
    "yapay zeka işsizlik Türkiye",
    "e ticaret Türkiye",
    "dijital okuryazarlık Türkiye",
]

# Duygu modeli. Model yüklenemezse kural tabanlı çalışır.
BERTURK_MODEL = "savasy/bert-base-turkish-sentiment-cased"

LDA_MIN_TOPIC = 3
LDA_MAX_TOPIC = 10
LDA_PASSES = 35
LDA_ITERATIONS = 450
TOP_N_WORDS = 30
TFIDF_TOP_N = 30
NGRAM_TOP_N = 25


# =============================================================================
# 2. STOPWORD + TEMİZLEME
# =============================================================================

CUSTOM_STOPWORDS = {
    "bir","bu","şu","o","ve","ile","da","de","ki","mi","mı","mu","mü","için",
    "gibi","kadar","ama","fakat","ancak","çünkü","veya","hem","ise","diye",
    "çok","daha","en","az","pek","epey","gayet","tam","hiç","bile","zaten",
    "artık","sadece","yani","ayrıca","öyle","böyle","şöyle","her","tüm","bütün",
    "bazı","birkaç","birçok","ne","neden","niçin","nasıl","hangi","kim","kimi",
    "kime","kimden","nerede","nereden","nereye","kaç","var","yok","değil",
    "olan","olarak","oldu","olup","olur","olsun","olmak","etmek","yapmak",
    "edildi","yapıldı","etti","olanlar","şey","şeyi","şeyler",
    "ben","sen","biz","siz","onlar","beni","bana","benim","seni","sana","senin",
    "bizi","bize","bizim","sizi","size","sizin","onu","ona","onun","onları",
    "onlara","onların","burada","orada","buraya","oraya","şuan","şimdi",
    "https","http","www","com","tr","co","youtube","video","yorum","kanal",
}

# YouTube özel gürültüleri: kişi/kanal adı, argo, anlamsız parçalar
NOISE_TERMS = {
    "jax","zooble","kinger","gang","gangleyi","ap","cain","lan","abi","abla",
    "kanka","knk","aga","bey","hanım","allah","inşallah","maşallah","amin",
    "me","ni","do","ol","lol","haha","ahaha","ahah","xd","sj","jsjs","random",
    "deha","sirk","sirki","toilet"
}

STOPWORDS = set()
STOPWORDS |= NLTK_STOPWORDS
STOPWORDS |= CUSTOM_STOPWORDS
STOPWORDS |= NOISE_TERMS

EMOJI_SENTIMENT_MAP = {
    "😂": " pozitif ",
    "🤣": " pozitif ",
    "😊": " pozitif ",
    "😍": " pozitif ",
    "❤️": " pozitif ",
    "❤": " pozitif ",
    "👍": " pozitif ",
    "👏": " pozitif ",
    "🔥": " pozitif ",
    "😡": " negatif ",
    "🤬": " negatif ",
    "😢": " negatif ",
    "😭": " negatif ",
    "👎": " negatif ",
    "💔": " negatif ",
    "😨": " negatif ",
    "😱": " negatif ",
}

def replace_emoji_with_signal(text: str) -> str:
    text = str(text)
    for emoji, repl in EMOJI_SENTIMENT_MAP.items():
        text = text.replace(emoji, repl)
    return text

def clean_text(text: str, use_stemmer: bool = True) -> str:
    text = replace_emoji_with_signal(text)
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+|#\w+", " ", text)
    text = re.sub(r"[^a-zA-ZçğıöşüÇĞİÖŞÜ\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = []
    for token in text.split():
        token = token.strip()
        if len(token) < 3:
            continue
        if token in STOPWORDS:
            continue
        if token.isdigit():
            continue
        tokens.append(token)

    if use_stemmer and USE_STEMMER:
        tokens = [TR_STEMMER.stemWord(t) for t in tokens]

    tokens = [t for t in tokens if len(t) > 2 and t not in STOPWORDS]
    return " ".join(tokens)

def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.dropna(subset=["yorum"], inplace=True)
    df["yorum"] = df["yorum"].astype(str)
    df["begeni"] = pd.to_numeric(df.get("begeni", 0), errors="coerce").fillna(0).astype(int)
    df["yanit_sayisi"] = pd.to_numeric(df.get("yanit_sayisi", 0), errors="coerce").fillna(0).astype(int)

    if "tarih" in df.columns:
        df["tarih"] = pd.to_datetime(df["tarih"], errors="coerce", utc=True)

    df.drop_duplicates(subset=["yorum"], inplace=True)
    df["temiz_yorum"] = df["yorum"].apply(clean_text)
    df = df[df["temiz_yorum"].str.strip() != ""].copy()
    df.drop_duplicates(subset=["temiz_yorum"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df


# =============================================================================
# 3. YOUTUBE API İLE ~8.000 VERİ TOPLAMA
# =============================================================================

def youtube_client():
    if not YOUTUBE_API_KEY:
        raise ValueError("YOUTUBE_API_KEY boş. Ortam değişkeni olarak ver veya dosyada YOUTUBE_API_KEY alanına yaz.")
    return build("youtube", "v3", developerKey=YOUTUBE_API_KEY)

def search_video_ids(youtube, keyword, max_videos=50):
    videos = []
    page_token = None

    while len(videos) < max_videos:
        try:
            resp = youtube.search().list(
                q=keyword,
                part="id,snippet",
                type="video",
                relevanceLanguage="tr",
                regionCode="TR",
                order="relevance",
                maxResults=min(50, max_videos - len(videos)),
                pageToken=page_token,
            ).execute()

            for item in resp.get("items", []):
                vid = item["id"].get("videoId")
                title = item["snippet"].get("title", "")
                if vid:
                    videos.append((vid, title, keyword))

            page_token = resp.get("nextPageToken")
            if not page_token:
                break

            time.sleep(REQUEST_SLEEP)

        except HttpError as e:
            print(f"[UYARI] Search hatası: {keyword} | {e}")
            break

    return videos

def get_comments_for_video(youtube, video_id, video_title, keyword, max_comments=250):
    rows = []
    page_token = None

    while len(rows) < max_comments:
        try:
            resp = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=min(100, max_comments - len(rows)),
                textFormat="plainText",
                order="relevance",
                pageToken=page_token,
            ).execute()

            for item in resp.get("items", []):
                snip = item["snippet"]["topLevelComment"]["snippet"]
                rows.append({
                    "video_id": video_id,
                    "video_baslik": video_title,
                    "keyword": keyword,
                    "yorum": snip.get("textDisplay", ""),
                    "begeni": snip.get("likeCount", 0),
                    "yanit_sayisi": item["snippet"].get("totalReplyCount", 0),
                    "tarih": snip.get("publishedAt", None),
                    "kanal": snip.get("authorDisplayName", ""),
                })

            page_token = resp.get("nextPageToken")
            if not page_token:
                break

            time.sleep(REQUEST_SLEEP)

        except HttpError as e:
            # Yorum kapalı, kota, izin vb.
            print(f"[UYARI] Yorum çekilemedi: {video_id} | {e}")
            break

    return rows

def collect_data(target_comments=TARGET_COMMENTS):
    if os.path.exists(RAW_CSV):
        df_old = pd.read_csv(RAW_CSV)
        if len(df_old) >= target_comments:
            print(f"[*] Mevcut ham veri yeterli: {len(df_old)} yorum")
            return df_old
        print(f"[*] Mevcut ham veri var ama hedefin altında: {len(df_old)} / {target_comments}")
        all_rows = df_old.to_dict("records")
        seen_comments = set(df_old["yorum"].astype(str).tolist()) if "yorum" in df_old.columns else set()
    else:
        all_rows = []
        seen_comments = set()

    youtube = youtube_client()

    print("[*] Video aramaları yapılıyor...")
    video_pool = []
    seen_video = set()

    for kw in KEYWORDS:
        vids = search_video_ids(youtube, kw, MAX_VIDEOS_PER_KEYWORD)
        for vid, title, keyword in vids:
            if vid not in seen_video:
                seen_video.add(vid)
                video_pool.append((vid, title, keyword))
        print(f"  {kw}: toplam video havuzu {len(video_pool)}")
        time.sleep(REQUEST_SLEEP)

    print(f"[*] Toplam benzersiz video: {len(video_pool)}")
    print("[*] Yorumlar çekiliyor...")

    for vid, title, kw in tqdm(video_pool):
        if len(all_rows) >= target_comments:
            break

        rows = get_comments_for_video(
            youtube,
            video_id=vid,
            video_title=title,
            keyword=kw,
            max_comments=MAX_COMMENTS_PER_VIDEO,
        )

        for r in rows:
            comment = str(r.get("yorum", "")).strip()
            if comment and comment not in seen_comments:
                seen_comments.add(comment)
                all_rows.append(r)

        if len(all_rows) % 500 < MAX_COMMENTS_PER_VIDEO:
            pd.DataFrame(all_rows).to_csv(RAW_CSV, index=False, encoding="utf-8-sig")

        print(f"  Toplam yorum: {len(all_rows)} / {target_comments}", end="\r")
        time.sleep(REQUEST_SLEEP)

    df = pd.DataFrame(all_rows)
    if "tarih" in df.columns:
        df["tarih"] = pd.to_datetime(df["tarih"], errors="coerce", utc=True)
    df.drop_duplicates(subset=["yorum"], inplace=True)
    df.to_csv(RAW_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✓ Ham veri kaydedildi: {len(df)} yorum → {RAW_CSV}")
    return df


# =============================================================================
# 4. DUYGU ANALİZİ
# =============================================================================

POSITIVE_WORDS = {
    "iyi","güzel","harika","mükemmel","başarılı","süper","teşekkür","olumlu",
    "faydalı","yararlı","gelişme","ilerleme","umut","verimli","etkili","memnun",
    "mutlu","destekliyorum","avantaj","fırsat","başarı","kaliteli","modern",
    "güvenli","doğru","sevindim","etkileyici","kolay","pozitif"
}

NEGATIVE_WORDS = {
    "kötü","berbat","rezalet","sorun","problem","hata","yavaş","zor","imkansız",
    "korkunç","üzücü","başarısız","yetersiz","eksik","geri","eşitsizlik",
    "uçurum","engel","dezavantaj","kaygı","endişe","tehlike","yanlış","haksız",
    "mahrum","pahalı","zorluk","sıkıntı","bozuk","çöktü","negatif","olumsuz"
}

def rule_based_sentiment(cleaned_text: str) -> tuple:
    tokens = set(str(cleaned_text).split())
    p = len(tokens & POSITIVE_WORDS)
    n = len(tokens & NEGATIVE_WORDS)
    if p > n:
        return "Pozitif", min(0.99, 0.55 + 0.12 * (p - n))
    if n > p:
        return "Negatif", min(0.99, 0.55 + 0.12 * (n - p))
    return "Nötr", 0.50

def run_sentiment(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if TRANSFORMERS_OK:
        try:
            print("[*] BERTurk duygu modeli yükleniyor...")
            tokenizer = AutoTokenizer.from_pretrained(BERTURK_MODEL)
            model = AutoModelForSequenceClassification.from_pretrained(BERTURK_MODEL)
            sentiment_pipe = pipeline(
                "text-classification",
                model=model,
                tokenizer=tokenizer,
                truncation=True,
                max_length=256,
            )

            label_map = {
                "positive": "Pozitif",
                "negative": "Negatif",
                "neutral": "Nötr",
                "LABEL_0": "Negatif",
                "LABEL_1": "Nötr",
                "LABEL_2": "Pozitif",
            }

            labels, scores = [], []
            texts = df["temiz_yorum"].astype(str).tolist()
            batch_size = 32

            for i in tqdm(range(0, len(texts), batch_size)):
                batch = texts[i:i + batch_size]
                preds = sentiment_pipe(batch)
                for pred in preds:
                    labels.append(label_map.get(str(pred["label"]).lower(), label_map.get(pred["label"], "Nötr")))
                    scores.append(round(float(pred["score"]), 4))

            df["duygu"] = labels
            df["duygu_skor"] = scores

        except Exception as e:
            print(f"[UYARI] BERTurk çalışmadı, kural tabanlıya geçiliyor: {e}")
            rb = df["temiz_yorum"].apply(rule_based_sentiment)
            df["duygu"] = rb.apply(lambda x: x[0])
            df["duygu_skor"] = rb.apply(lambda x: x[1])
    else:
        rb = df["temiz_yorum"].apply(rule_based_sentiment)
        df["duygu"] = rb.apply(lambda x: x[0])
        df["duygu_skor"] = rb.apply(lambda x: x[1])

    print("\nDuygu dağılımı:")
    print(df["duygu"].value_counts().to_string())
    return df


# =============================================================================
# 5. ANALİZLER
# =============================================================================

def word_frequency(df: pd.DataFrame, top_n=TOP_N_WORDS):
    words = " ".join(df["temiz_yorum"].dropna().astype(str)).split()
    words = [w for w in words if w not in STOPWORDS and len(w) > 2]
    return Counter(words).most_common(top_n)

def ngram_analysis(df: pd.DataFrame, n=2, top_n=NGRAM_TOP_N):
    vec = CountVectorizer(
        ngram_range=(n, n),
        max_features=6000,
        min_df=3,
        max_df=0.80,
    )
    X = vec.fit_transform(df["temiz_yorum"])
    totals = X.sum(axis=0).A1
    terms = vec.get_feature_names_out()
    result = sorted(zip(terms, totals), key=lambda x: -x[1])[:top_n]
    return result

def tfidf_analysis(df: pd.DataFrame, top_n=TFIDF_TOP_N):
    vec = TfidfVectorizer(
        max_features=8000,
        ngram_range=(1, 2),
        min_df=4,
        max_df=0.75,
        sublinear_tf=True,
    )
    X = vec.fit_transform(df["temiz_yorum"])
    means = X.mean(axis=0).A1
    terms = vec.get_feature_names_out()

    result = pd.DataFrame({"terim": terms, "tfidf_ortalama": means})
    result["terim"] = result["terim"].astype(str)
    result = result[~result["terim"].isin(NOISE_TERMS)]
    result = result[result["terim"].str.len() > 2]
    return result.sort_values("tfidf_ortalama", ascending=False).head(top_n)

def choose_lda_topic_count(tokenized):
    dictionary = corpora.Dictionary(tokenized)
    dictionary.filter_extremes(no_below=4, no_above=0.80)
    corpus = [dictionary.doc2bow(doc) for doc in tokenized]

    scores = []
    models = {}

    for k in range(LDA_MIN_TOPIC, LDA_MAX_TOPIC + 1):
        lda = LdaModel(
            corpus=corpus,
            id2word=dictionary,
            num_topics=k,
            passes=20,
            iterations=250,
            random_state=42,
            alpha="auto",
            eta="auto",
        )
        coherence = CoherenceModel(
            model=lda,
            texts=tokenized,
            dictionary=dictionary,
            coherence="c_v",
        ).get_coherence()

        scores.append({"konu_sayisi": k, "coherence": coherence})
        models[k] = lda

    scores_df = pd.DataFrame(scores)
    best_k = int(scores_df.sort_values("coherence", ascending=False).iloc[0]["konu_sayisi"])
    return best_k, scores_df, corpus, dictionary

def run_lda(df: pd.DataFrame):
    tokenized = [
        [w for w in str(text).split() if w not in STOPWORDS and len(w) > 2]
        for text in df["temiz_yorum"].tolist()
    ]

    best_k, scores_df, corpus, dictionary = choose_lda_topic_count(tokenized)

    lda_model = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=best_k,
        passes=LDA_PASSES,
        iterations=LDA_ITERATIONS,
        random_state=42,
        alpha="auto",
        eta="auto",
    )

    return lda_model, corpus, dictionary, scores_df

def lda_dominant_topic(lda_model, corpus, df):
    df = df.copy()
    topics, scores = [], []

    for bow in corpus:
        dist = lda_model.get_document_topics(bow)
        if dist:
            top_topic, top_score = max(dist, key=lambda x: x[1])
            topics.append(top_topic + 1)
            scores.append(round(float(top_score), 4))
        else:
            topics.append(0)
            scores.append(0.0)

    df["lda_konu"] = topics
    df["lda_skor"] = scores
    return df

def save_lda_topics(lda_model):
    rows = []
    for t in range(lda_model.num_topics):
        pairs = lda_model.show_topic(t, topn=15)
        rows.append({
            "konu_no": t + 1,
            "konu_basligi": auto_topic_label([p[0] for p in pairs]),
            "kelimeler": ", ".join([p[0] for p in pairs]),
            "agirliklar": ", ".join([str(round(float(p[1]), 4)) for p in pairs]),
        })

    out = pd.DataFrame(rows)
    out.to_csv(os.path.join(OUTPUT_DIR, "lda_konular.csv"), index=False, encoding="utf-8-sig")
    return out

def auto_topic_label(words):
    words = set(words)

    rules = [
        ("Dijital Eğitim", {"eğitim","öğrenci","okul","ders","uzaktan","öğretmen"}),
        ("Yapay Zekâ ve Teknoloji", {"yapay","zeka","teknoloji","robot","otomasyon"}),
        ("E-Devlet ve Kamu Hizmetleri", {"devlet","edevlet","hizmet","kamu","sistem"}),
        ("Dijital Uçurum ve Erişim", {"internet","erişim","uçurum","bölge","köy","gelir"}),
        ("Ekonomi ve İş Gücü", {"iş","ekonomi","para","meslek","çalışma","işsizlik"}),
        ("Genel Toplumsal Algı", set()),
    ]

    for label, keys in rules:
        if len(words & keys) >= 1:
            return label
    return "Genel Toplumsal Algı"

def representative_comments(df, n=4):
    rows = []

    for duygu in ["Pozitif", "Negatif", "Nötr"]:
        subset = df[df["duygu"] == duygu].copy()
        if subset.empty:
            continue

        # Hem beğeni hem duygu skoru yüksek olanları seç
        subset["temsil_skoru"] = (
            np.log1p(pd.to_numeric(subset["begeni"], errors="coerce").fillna(0))
            + pd.to_numeric(subset["duygu_skor"], errors="coerce").fillna(0)
        )
        subset = subset.sort_values("temsil_skoru", ascending=False).head(n)

        for _, r in subset.iterrows():
            rows.append({
                "duygu": duygu,
                "yorum": str(r["yorum"])[:260],
                "begeni": int(r.get("begeni", 0)),
                "keyword": r.get("keyword", ""),
                "duygu_skor": r.get("duygu_skor", 0),
                "lda_konu": r.get("lda_konu", ""),
            })

    return pd.DataFrame(rows)


# =============================================================================
# 6. GÖRSELLEŞTİRME
# =============================================================================

PALETTE = {"Pozitif": "#16A34A", "Nötr": "#D97706", "Negatif": "#DC2626"}
C5 = ["#2563EB", "#DC2626", "#16A34A", "#D97706", "#7C3AED", "#0891B2", "#DB2777"]

def savefig(name):
    path = os.path.join(OUTPUT_DIR, name)
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.close()
    print(f"✓ {path}")

def plot_sentiment(df):
    counts = df["duygu"].value_counts().reindex(["Pozitif", "Nötr", "Negatif"]).dropna()
    colors = [PALETTE.get(i, "#999") for i in counts.index]

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(counts.index, counts.values, color=colors)
    total = counts.sum()

    for b in bars:
        v = int(b.get_height())
        ax.text(
            b.get_x() + b.get_width() / 2,
            b.get_height() + max(total * 0.01, 1),
            f"{v}\n%{v/total*100:.1f}",
            ha="center",
            fontsize=10,
            fontweight="bold",
        )

    ax.set_title("YouTube Yorumları Duygu Dağılımı", fontweight="bold")
    ax.set_ylabel("Yorum Sayısı")
    savefig("01_duygu_dagilimi.png")

def plot_word_freq(freq):
    if not freq:
        return
    words, counts = zip(*freq)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(list(words)[::-1], list(counts)[::-1], color=C5[4])
    ax.set_title("En Sık Kullanılan Anlamlı Kelimeler", fontweight="bold")
    ax.set_xlabel("Frekans")
    savefig("02_kelime_frekansi.png")

def plot_wordcloud(df):
    text = " ".join(df["temiz_yorum"].dropna().astype(str))
    if not text.strip():
        return

    wc = WordCloud(
        width=1400,
        height=700,
        background_color="white",
        max_words=180,
        colormap="viridis",
    ).generate(text)

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title("Kelime Bulutu", fontweight="bold", fontsize=14)
    savefig("03_kelime_bulutu.png")

def plot_ngrams(bigrams, trigrams):
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    for ax, data, title in zip(axes, [bigrams, trigrams], ["Bigram Analizi", "Trigram Analizi"]):
        if not data:
            ax.axis("off")
            continue
        labels, vals = zip(*data)
        ax.barh(list(labels)[::-1], list(vals)[::-1], color=C5[1])
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Frekans")
        ax.tick_params(axis="y", labelsize=8)

    plt.tight_layout()
    savefig("04_ngram_analizi.png")

def plot_tfidf(tfidf_df):
    if tfidf_df.empty:
        return
    fig, ax = plt.subplots(figsize=(11, 7))
    ax.barh(tfidf_df["terim"][::-1], tfidf_df["tfidf_ortalama"][::-1], color=C5[2])
    ax.set_title("TF-IDF – Ayırt Edici Terimler", fontweight="bold")
    ax.set_xlabel("Ortalama TF-IDF Skoru")
    ax.tick_params(axis="y", labelsize=8)
    savefig("05_tfidf_analizi.png")

def plot_lda_topics(lda_model):
    n_topics = lda_model.num_topics
    ncols = 2
    nrows = int(np.ceil(n_topics / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 4))
    axes = np.array(axes).flatten()

    for t in range(n_topics):
        pairs = lda_model.show_topic(t, topn=10)
        words = [p[0] for p in pairs]
        weights = [p[1] for p in pairs]
        label = auto_topic_label(words)

        axes[t].barh(words[::-1], weights[::-1], color=C5[t % len(C5)])
        axes[t].set_title(f"Konu {t+1}: {label}", fontweight="bold")
        axes[t].set_xlabel("Ağırlık")
        axes[t].tick_params(axis="y", labelsize=8)

    for i in range(n_topics, len(axes)):
        axes[i].axis("off")

    plt.suptitle("LDA Konu Modelleme", fontweight="bold", fontsize=14)
    plt.tight_layout()
    savefig("06_lda_konular.png")

def plot_lda_distribution(df):
    if "lda_konu" not in df.columns:
        return
    counts = df["lda_konu"].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar([f"Konu {int(k)}" for k in counts.index], counts.values, color=C5[:len(counts)])
    ax.set_title("LDA – Konu Başına Yorum Sayısı", fontweight="bold")
    ax.set_ylabel("Yorum Sayısı")
    savefig("07_lda_yorum_dagilimi.png")

def plot_time_series(df):
    if "tarih" not in df.columns or df["tarih"].isna().all():
        return

    df2 = df.copy()
    df2["ay"] = pd.to_datetime(df2["tarih"], errors="coerce", utc=True).dt.to_period("M").astype(str)
    monthly_sent = df2.groupby(["ay", "duygu"]).size().unstack(fill_value=0)

    fig, ax = plt.subplots(figsize=(13, 6))
    for duygu in ["Pozitif", "Nötr", "Negatif"]:
        if duygu in monthly_sent.columns:
            ax.plot(monthly_sent.index, monthly_sent[duygu], marker="o", label=duygu, color=PALETTE.get(duygu))

    ax.set_title("Aylık Duygu Eğilimi", fontweight="bold")
    ax.set_ylabel("Yorum Sayısı")
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.legend()
    savefig("08_zaman_serisi.png")

def plot_top_engagement(df, top_n=15):
    if "begeni" not in df.columns:
        return

    top = df.nlargest(top_n, "begeni")[["yorum", "begeni", "duygu"]].copy()
    if top.empty:
        return
    top["yorum_kisa"] = top["yorum"].astype(str).str[:58] + "…"
    colors = [PALETTE.get(d, "#999") for d in top["duygu"]]

    fig, ax = plt.subplots(figsize=(12, 7))
    ax.barh(top["yorum_kisa"][::-1], top["begeni"][::-1], color=colors[::-1])
    ax.set_title("En Çok Beğeni Alan Yorumlar", fontweight="bold")
    ax.set_xlabel("Beğeni Sayısı")
    ax.tick_params(axis="y", labelsize=8)
    savefig("09_top_engagement.png")

def plot_sentiment_heatmap(df):
    if "keyword" not in df.columns:
        return
    pivot = df.groupby(["keyword", "duygu"]).size().unstack(fill_value=0)
    fig, ax = plt.subplots(figsize=(11, 7))
    sns.heatmap(pivot, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_title("Anahtar Kelime × Duygu Isı Haritası", fontweight="bold")
    ax.set_xlabel("Duygu")
    ax.set_ylabel("Anahtar Kelime")
    savefig("10_duygu_isi_haritasi.png")

def plot_coherence(scores_df):
    if scores_df.empty:
        return
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(scores_df["konu_sayisi"], scores_df["coherence"], marker="o", color=C5[0])
    ax.set_title("LDA Coherence Skoru ile Konu Sayısı Seçimi", fontweight="bold")
    ax.set_xlabel("Konu Sayısı")
    ax.set_ylabel("Coherence Score")
    ax.grid(alpha=0.25)
    savefig("11_lda_coherence.png")


# =============================================================================
# 7. ANA AKIŞ
# =============================================================================

def main():
    print("=" * 70)
    print("YouTube NLP – Türkiye Dijitalleşme Algısı | İyileştirilmiş")
    print("=" * 70)

    print("\n[*] Veri toplama...")
    df_raw = collect_data(TARGET_COMMENTS)

    if df_raw.empty:
        print("[HATA] Veri yok.")
        return

    print("\n[*] Ön işleme...")
    df = preprocess(df_raw)
    df.to_csv(CLEAN_CSV, index=False, encoding="utf-8-sig")
    print(f"✓ Temiz veri: {len(df)} yorum → {CLEAN_CSV}")

    print("\n[*] Duygu analizi...")
    df = run_sentiment(df)

    print("\n[*] Kelime frekansı...")
    freq = word_frequency(df)

    print("[*] N-gram...")
    bigrams = ngram_analysis(df, n=2)
    trigrams = ngram_analysis(df, n=3)

    print("[*] TF-IDF...")
    tfidf_df = tfidf_analysis(df)
    tfidf_df.to_csv(os.path.join(OUTPUT_DIR, "tfidf_sonuclar.csv"), index=False, encoding="utf-8-sig")

    print("[*] LDA + Coherence ile en uygun konu sayısı...")
    lda_model, corpus, dictionary, coherence_df = run_lda(df)
    coherence_df.to_csv(os.path.join(OUTPUT_DIR, "lda_coherence.csv"), index=False, encoding="utf-8-sig")
    df = lda_dominant_topic(lda_model, corpus, df)
    lda_topics_df = save_lda_topics(lda_model)

    print("\nSeçilen konu sayısı:", lda_model.num_topics)
    print(lda_topics_df[["konu_no", "konu_basligi", "kelimeler"]].to_string(index=False))

    print("\n[*] Temsilî yorumlar...")
    rep_df = representative_comments(df)
    rep_df.to_csv(os.path.join(OUTPUT_DIR, "temsili_yorumlar.csv"), index=False, encoding="utf-8-sig")

    print("[*] Görseller...")
    plot_sentiment(df)
    plot_word_freq(freq)
    plot_wordcloud(df)
    plot_ngrams(bigrams, trigrams)
    plot_tfidf(tfidf_df)
    plot_lda_topics(lda_model)
    plot_lda_distribution(df)
    plot_time_series(df)
    plot_top_engagement(df)
    plot_sentiment_heatmap(df)
    plot_coherence(coherence_df)

    try:
        vis = gensimvis.prepare(lda_model, corpus, dictionary, mds="mmds")
        pyLDAvis.save_html(vis, os.path.join(OUTPUT_DIR, "lda_interaktif.html"))
        print("✓ gorseller/lda_interaktif.html")
    except Exception as e:
        print(f"[UYARI] pyLDAvis oluşturulamadı: {e}")

    df.to_csv(FINAL_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✓ Nihai dosya: {FINAL_CSV}")
    print(f"✓ Toplam temiz yorum: {len(df)}")
    print("=" * 70)

if __name__ == "__main__":
    main()


YouTube NLP – Türkiye Dijitalleşme Algısı | İyileştirilmiş

[*] Veri toplama...
[*] Mevcut ham veri yeterli: 8008 yorum

[*] Ön işleme...
✓ Temiz veri: 7384 yorum → youtube_temiz.csv

[*] Duygu analizi...

Duygu dağılımı:
duygu
Nötr       5258
Pozitif    1681
Negatif     445

[*] Kelime frekansı...
[*] N-gram...
[*] TF-IDF...
[*] LDA + Coherence ile en uygun konu sayısı...

Seçilen konu sayısı: 6
 konu_no                konu_basligi                                                                                                     kelimeler
       1        Genel Toplumsal Algı   negatif, aşak, yukar, iniyor, çıkıyor, dede, rahmet, cennet, gidiyor, gül, sepet, kirli, müzik, meka, drone
       2 E-Devlet ve Kamu Hizmetleri                    türki, ülke, şarkı, dünya, türk, nato, zama, yol, devlet, abd, ver, çin, büyük, savaş, geç
       3              Dijital Eğitim                         çocuk, gel, yap, yıl, okul, ada, gün, sonra, lütfe, hal, bak, keşke, yaş, diyor, aynı
       4    